# Stage 6 — Competing Risks & Recurrent Events

Removes two Stage-4 biases: non-injury exits treated as censoring, and
time-to-first-injury only. Requires `person_period.parquet` (Stage 3).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

## 1. Cause-specific hazards + cumulative incidence
Two calibrated cause-specific models (injury vs non-injury exit), combined
into a CIF over a one-season horizon. Calibration (not AUC) governs here.

In [ ]:
from nba_injury.model_competing_risks import run
out = run(use_pydts=False)
out['cif']

## 2. CIF curve over the season horizon
Injury incidence, exit incidence, and survival should sum toward 1 at every t.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from nba_injury.model_competing_risks import (
    load_table, build_recurrent_table, code_causes, temporal_split,
    _fit_cause_specific, cumulative_incidence, CAUSE_INJURY, CAUSE_EXIT,
    TIER1_FEATURES)
df = code_causes(build_recurrent_table(load_table()))
tr, te, *_ = temporal_split(df)
feats = TIER1_FEATURES + ['weeks_since_last_injury']
_, p_i, _ = _fit_cause_specific(tr, te, CAUSE_INJURY, feats)
_, p_e, _ = _fit_cause_specific(tr, te, CAUSE_EXIT, feats)
Ts = range(1, 27)
ci = [cumulative_incidence(p_i, p_e, t)['cif_injury_final'] for t in Ts]
ce = [cumulative_incidence(p_i, p_e, t)['cif_exit_final'] for t in Ts]
s = [cumulative_incidence(p_i, p_e, t)['survival_final'] for t in Ts]
plt.stackplot(list(Ts), ci, ce, s, labels=['CIF injury', 'CIF exit', 'survival'])
plt.xlabel('weeks into season'); plt.ylabel('probability'); plt.legend(loc='center left')
plt.title('Competing-risks cumulative incidence'); plt.show()

## 3. PyDTS cross-check (optional, degrades gracefully)
The academic competing-risks package; falls back to the sklearn anchor on any
rough edge.

In [ ]:
out2 = run(use_pydts=True)
print('PyDTS:', out2.get('pydts'))

## 4. Gate 6

In [ ]:
from nba_injury.gate6_report import main as gate6
gate6()